Copyright (c) 2019-2026, Members of the Simons Observatory collaboration.
Please refer to the [LICENSE](./LICENSE) file in the root of this repository.

These commands make sure that matplotlib is configured correctly to run in 
docker. It makes matplotlib use a non-graphical backend, but allows you to 
still show plots inline in a notebook. This bypasses docker graphical issues
that usually pop up, and will be needed in most notebooks.

In [12]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

# Sodetlib Config Overview

In [2]:
from sodetlib.det_config import DetConfig
from pprint import pprint

Rogue/pyrogue version v6.13.0. https://github.com/slaclab/rogue


## Guide to configurations files

In the current system deployed for SO, there are multiple configuration files that need to be present. These are typically stored in a directory defined by the `$SMURF_CONFIG_DIR` environment variable on the server node.

The config files are:
- `sys_config.yml`: high-level description of the system, including populated slots, IP addresses, and docker tags to use for the software. Also includes paths to subsequent configs. Used by `sodetlib` and `jackhammer` to start and configure the SMuRF system.
- Device config: used by `sodetlib` to store information about the state of the system, per slot. This includes amplifier biases, tuning data, active bands, etc. `sodetlib` operations will modify this file to store the latest state (usually accepting and `update_config` boolean argument).
- `pysmurf` config: used by `pysmurf` (lower-level library) to initialise the hardware with default register values. Specific to an individual system and shouldn't need to be edited once deployed.

Typically a user will interact with the `sys_config.yml` file to configure a system, and the device configs via `sodetlib` operations.

More details can be found in the [`sodetlib` docs](https://sodetlib.readthedocs.io/en/latest/configs.html). Examples from the [SO deployment configs](https://github.com/simonsobs/ocs-deployment-configs/tree/main/lat/smurf-so8-lat) are also available.

In `python`, the `DetConfig` object is used to load system, device, and pysmurf configurations. By default, it will first load the system configuration from `$SMURF_CONFIG_DIR/sys_config.yml`. This config file should contain the device and pysmurf config files to use for each smurf slot, which the `DetConfig` system will then load.

Assuming the default location for `sys_config.yml`, only the slot we wish to access needs to be specified.

## Loading the configuration for a given slot

In [3]:
cfg = DetConfig()
cfg.load_config_files(slot=4)

The system config and device config, can be accessed at `cfg.sys` and `cfg.dev` respectively.

In [4]:
print("Sys config\n" + 13*"-")
pprint(cfg.sys)

Sys config
-------------
{'comm_type': 'pcie',
 'crate_id': 1,
 'docker_env': {'CB_HOST': 'pc98970',
                'OCS_TAG': 'v0.11.3',
                'SOCS_TAG': 'v0.5.8',
                'STREAMER_TAG': 'v0.4.4-6-g7b2f5bb'},
 'g3_dir': '/data/so/timestreams',
 'init_fan_level': 30,
 'max_fan_level': 50,
 'meta_register_file': '$OCS_CONFIG_DIR/meta_registers.yaml',
 'min_fan_level': 10,
 'shelf_manager': 'shm-smrf-sp01',
 'slot_order': [4, 6],
 'slots': {'SLOT[4]': {'device_config': '$OCS_CONFIG_DIR/device_configs/dev_cfg_s4.yaml',
                       'pysmurf_config': '$OCS_CONFIG_DIR/pysmurf_config/experiment_b33_rfc1-2_C05-40_s4_2xLB_extref.cfg',
                       'stream_id': 'crate1slot4',
                       'stream_port': 4534},
           'SLOT[6]': {'device_config': '$OCS_CONFIG_DIR/device_configs/dev_cfg_s6.yaml',
                       'pysmurf_config': '$OCS_CONFIG_DIR/pysmurf_config/experiment_b33_rfc1-2_C05-40_s4_2xLB_extref.cfg',
                       's

Here we see that the system is configured for slots 4 and 6 in crate 1, and can read off the docker tags that will be used to start up the software and firmware.

The device config is split into experiment config, or `exp`, which contains general config info about the device,
`bands` which contains info about the 8 bands on the slot, and `bias_groups` which contains config info about the 12 bias groups.

In [5]:
print("Exp\n"+5*"-")
pprint(cfg.dev.exp)
print("\nBand[0]\n" + 10*'-')
pprint(cfg.dev.bands[3])
print("\nBiasGroup[0]\n" + 10*'-')
pprint(cfg.dev.bias_groups[0])

Exp
-----
{'active_bands': [0, 1, 2, 3, 4, 5, 6, 7],
 'active_bgs': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 'amp_50k1_drain_current': 21.5,
 'amp_50k1_drain_current_tolerance': 0.2,
 'amp_50k1_drain_volt': 5.000003766174316,
 'amp_50k1_gate_volt': 0.0,
 'amp_50k1_gate_volt_max': 1,
 'amp_50k1_gate_volt_min': -1,
 'amp_50k1_init_gate_volt': -0.5,
 'amp_50k2_drain_current': 15.0,
 'amp_50k2_drain_current_tolerance': 0.2,
 'amp_50k2_drain_volt': 4,
 'amp_50k2_gate_volt': None,
 'amp_50k2_gate_volt_max': 1,
 'amp_50k2_gate_volt_min': -1,
 'amp_50k2_init_gate_volt': -0.5,
 'amp_50k_drain_current': 15.0,
 'amp_50k_drain_current_tolerance': 0.2,
 'amp_50k_gate_volt': None,
 'amp_50k_gate_volt_max': 1,
 'amp_50k_gate_volt_min': -1,
 'amp_50k_init_gate_volt': -0.5,
 'amp_enable_wait_time': 10.0,
 'amp_hemt1_drain_current': 4,
 'amp_hemt1_drain_current_tolerance': 0.2,
 'amp_hemt1_drain_volt': 0.5000014869880673,
 'amp_hemt1_gate_volt': 0.24997226408000003,
 'amp_hemt1_gate_volt_max': 0,
 'amp_

## Creating pysmurf instance

To create a pysmurf instance based on this configuration setup, simply run: 

In [6]:
S = cfg.get_smurf_control(dump_configs=True)

Connected to AMCc at localhost:9012
Dumping sodetlib configs to /data/smurf_data/20260609/crate1slot4/1781043249/config/1781043250


This will create a control object with the correct epics root, pysmurf config file, etc. and dump all of the config files to S.data_directory. Using this will also correctly set the pysmurf publisher ID correctly based on the crate id and slot number.